# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [7]:
# Write your code below.
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [16]:
import os
from glob import glob

# Write your code below.


# Get PRICE_DATA directory from env variable

# Load env variable into a Python variable
price_data_dir = os.getenv("PRICE_DATA")
print("PRICE_DATA =", price_data_dir)

# Use it in glob
#parquet_files = glob(os.path.join(price_data_dir, "**", "*.parquet"), recursive=True)
#print("Number of parquet files:", len(parquet_files))

parquet_files = glob(os.path.join(price_data_dir, "**/*.parquet"), recursive = True)
print("Number of parquet files:", len(parquet_files))

dd_px = dd.read_parquet(parquet_files).set_index("ticker")
print(dd_px.columns)





PRICE_DATA = ../../05_src/data/prices/
Number of parquet files: 2737
Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'Year'],
      dtype='object')


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [18]:
# Write your code below.
# Create lag features for Close and Adj_Close
dd_px['Close_lag_1'] = dd_px.groupby("ticker")['Close'].shift(1)
dd_px['Adj_Close_lag_1'] = dd_px.groupby("ticker")['Adj Close'].shift(1)

# Create returns
dd_px['returns'] = (dd_px['Close'] / dd_px['Close_lag_1']) - 1

# Create daily high-low range
dd_px['hi_lo_range'] = dd_px['High'] - dd_px['Low']

# Assign to dd_feat
dd_feat = dd_px

dd_feat.head()


C:\Users\ramee\AppData\Local\Temp\ipykernel_23292\1067367609.py:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_px['Close_lag_1'] = dd_px.groupby("ticker")['Close'].shift(1)
C:\Users\ramee\AppData\Local\Temp\ipykernel_23292\1067367609.py:4: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_px['Adj_Close_lag_1'] = dd_px.groupby("ticker")['Adj Close'].shift(1)


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
ticker,,,,,,,,,,,,,
ACN,2016-01-04,102.620003,102.650002,100.970001,101.830002,94.543030,2817000.0,ACN.csv,2016,NaN,NaN,NaN,1.680000
ACN,2016-01-05,101.970001,102.870003,101.470001,102.360001,95.035095,2409000.0,ACN.csv,2016,101.830002,94.543030,0.005205,1.400002
ACN,2016-01-06,100.809998,103.059998,100.540001,102.160004,94.849411,3134200.0,ACN.csv,2016,102.360001,95.035095,-0.001954,2.519997
ACN,2016-01-07,99.750000,100.839996,98.839996,99.160004,92.064087,3194700.0,ACN.csv,2016,102.160004,94.849411,-0.029366,2.000000
ACN,2016-01-08,99.480003,99.809998,98.000000,98.199997,91.172775,2330200.0,ACN.csv,2016,99.160004,92.064087,-0.009681,1.809998


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [19]:
# Write your code below.
# Convert Dask DataFrame to pandas
pd_feat = dd_feat.compute()

# Add 10-day moving average of returns (per ticker)
pd_feat['returns_ma_10'] = (
    pd_feat.groupby("ticker")['returns']
           .transform(lambda x: x.rolling(10).mean())
)

pd_feat.head(15)


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,returns_ma_10
ticker,,,,,,,,,,,,,,
ACN,2016-01-04,102.620003,102.650002,100.970001,101.830002,94.543030,2817000.0,ACN.csv,2016,NaN,NaN,NaN,1.680000,NaN
ACN,2016-01-05,101.970001,102.870003,101.470001,102.360001,95.035095,2409000.0,ACN.csv,2016,101.830002,94.543030,0.005205,1.400002,NaN
ACN,2016-01-06,100.809998,103.059998,100.540001,102.160004,94.849411,3134200.0,ACN.csv,2016,102.360001,95.035095,-0.001954,2.519997,NaN
ACN,2016-01-07,99.750000,100.839996,98.839996,99.160004,92.064087,3194700.0,ACN.csv,2016,102.160004,94.849411,-0.029366,2.000000,NaN
ACN,2016-01-08,99.480003,99.809998,98.000000,98.199997,91.172775,2330200.0,ACN.csv,2016,99.160004,92.064087,-0.009681,1.809998,NaN
ACN,2016-01-11,98.480003,99.720001,98.080002,99.230003,92.129082,2734400.0,ACN.csv,2016,98.199997,91.172775,0.010489,1.639999,NaN
ACN,2016-01-12,99.959999,101.290001,99.440002,101.010002,93.781708,2811000.0,ACN.csv,2016,99.230003,92.129082,0.017938,1.849998,NaN
ACN,2016-01-13,101.389999,101.750000,98.739998,99.260002,92.156929,3160200.0,ACN.csv,2016,101.010002,93.781708,-0.017325,3.010002,NaN
ACN,2016-01-14,99.120003,102.470001,98.910004,101.879997,94.589447,4285500.0,ACN.csv,2016,99.260002,92.156929,0.026395,3.559998,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No, it wasn’t strictly necessary. Dask does support .rolling() operations, so you could compute the moving average directly in Dask. But, rolling operations in Dask are more limited because Dask partitions data into chunks and  rolling window may not align across partitions.

If the dataset is small enough to fit in memory, converting to pandas is usually simpler and faster, because pandas has full-featured  rolling window functions.

If the dataset is very large , then it’s better to compute the rolling mean directly in Dask, so you can keep the computation distributed and avoid memory issues.But Dask only computes rolling windows correctly within each partition .

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.